In [ ]:
!pip install -q transformers peft bitsandbytes accelerate huggingface_hub

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from google.colab import userdata # Correct way to get secrets in Colab
from huggingface_hub import login

# Get Hugging Face token from Colab user data
hf_token = userdata.get("HF_TOKEN")
login(token=hf_token)

base_model_id = "meta-llama/Llama-3.2-1B-Instruct"
# Use your DPO adapter from notebook 2
dpo_adapter_id = "pranav6905/Llama-3.2-1B-DPO-DPOMix-Adapters"

print("Loading base model...")
# Load in fp16 for merging (NOT 4bit — can't merge quantized)
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    torch_dtype = torch.float16,
    device_map  = "cpu",           # merge on CPU to avoid VRAM limits
)

tokenizer = AutoTokenizer.from_pretrained(base_model_id)
tokenizer.pad_token = tokenizer.eos_token

print("Loading DPO LoRA adapters...")
model = PeftModel.from_pretrained(base_model, dpo_adapter_id)

print("Merging adapters into base model...")
model = model.merge_and_unload()   # this fuses LoRA weights permanently

print("✅ Merge complete!")
print(f"Model size: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B params")

In [ ]:
# This is the final deployable model
model.push_to_hub(
    "pranav6905/llama-1b-sft-dpo-final",
    token       = hf_token,
    private     = False,    # make it public so HF Spaces can access it
)
tokenizer.push_to_hub(
    "pranav6905/llama-1b-sft-dpo-final",
    token = hf_token,
)

print("✅ Final merged model on HuggingFace Hub!")
print("Model is ready for deployment")